In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format

pio.renderers.default = "browser"

In [ ]:
data = pd.read_csv('boston.csv', index_col=0)

print(f"Dataset Shape: {data.shape}")
print(f"Columns: {list(data.columns)}\n")
print(f"Any NaN values? {data.isna().values.any()}")
print(f"Any duplicates? {data.duplicated().values.any()}\n")

display(data.info())
display(data.describe())

In [ ]:
sns.displot(data['PRICE'], bins=50, aspect=2, kde=True, color='#2196f3')
plt.title(f'1970s Home Values in Boston. Average: ${(1000 * data.PRICE.mean()):,.2f}')
plt.xlabel('Price in $000s')
plt.ylabel('Nr. of Homes')
plt.show()

sns.displot(data.DIS, bins=50, aspect=2, kde=True, color='darkblue')
plt.title(f'Distance to Employment Centres. Average: {data.DIS.mean():.2f}')
plt.xlabel('Weighted Distance to 5 Boston Employment Centres')
plt.ylabel('Nr. of Homes')
plt.show()

sns.displot(data.RM, aspect=2, kde=True, color='#00796b')
plt.title(f'Distribution of Rooms in Boston. Average: {data.RM.mean():.2f}')
plt.xlabel('Average Number of Rooms')
plt.ylabel('Nr. of Homes')
plt.show()

plt.figure(figsize=(10, 5), dpi=200)
plt.hist(data['RAD'], bins=24, ec='black', color='#7b1fa2', rwidth=0.5)
plt.xlabel('Accessibility to Highways')
plt.ylabel('Nr. of Houses')
plt.show()

In [ ]:
river_access = data['CHAS'].value_counts()

bar = px.bar(
    x=['No', 'Yes'],
    y=river_access.values,
    color=river_access.values,
    color_continuous_scale=px.colors.sequential.haline,
    title='Next to Charles River?'
)

bar.update_layout(
    xaxis_title='Property Located Next to the River?', 
    yaxis_title='Number of Homes',
    coloraxis_showscale=False
)
bar.show()

In [ ]:


with sns.axes_style('darkgrid'):
    sns.jointplot(x=data['DIS'], y=data['NOX'], height=8, kind='scatter', color='deeppink', joint_kws={'alpha':0.5})
    plt.show()

    sns.jointplot(x=data.NOX, y=data.INDUS, height=7, color='darkgreen', joint_kws={'alpha':0.5})
    plt.show()

    sns.jointplot(x=data['LSTAT'], y=data['RM'], height=7, color='orange', joint_kws={'alpha':0.5})
    plt.show()

    sns.jointplot(x=data.LSTAT, y=data.PRICE, height=7, color='crimson', joint_kws={'alpha':0.5})
    plt.show()

with sns.axes_style('whitegrid'):
    sns.jointplot(x=data.RM, y=data.PRICE, height=7, color='darkblue', joint_kws={'alpha':0.5})
    plt.show()

In [ ]:
target = data['PRICE']
features = data.drop('PRICE', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    features, 
    target, 
    test_size=0.2, 
    random_state=10
)

train_pct = 100 * len(X_train) / len(features)
test_pct = 100 * X_test.shape[0] / features.shape[0]

print(f'Training data is {train_pct:.2f}% of the total data.')
print(f'Test data makes up the remaining {test_pct:.2f}%.')

In [ ]:
regr = LinearRegression()
regr.fit(X_train, y_train)
rsquared = regr.score(X_train, y_train)

print(f'Training data R-squared: {rsquared:.4f}')

regr_coef = pd.DataFrame(data=regr.coef_, index=X_train.columns, columns=['Coefficient'])
display(regr_coef)

premium = regr_coef.loc['RM'].values[0] * 1000  
print(f'The price premium for having an extra room is ${premium:,.2f}')

In [ ]:
predicted_vals = regr.predict(X_train)
residuals = (y_train - predicted_vals)

plt.figure(dpi=100)
plt.scatter(x=y_train, y=predicted_vals, c='indigo', alpha=0.6)
plt.plot(y_train, y_train, color='cyan')
plt.title('Actual vs Predicted Prices: $y_i$ vs $\hat{y}_i$', fontsize=17)
plt.xlabel('Actual prices $000s $y_i$', fontsize=14)
plt.ylabel('Predicted prices $000s \hat{y}_i$', fontsize=14)
plt.show()

plt.figure(dpi=100)
plt.scatter(x=predicted_vals, y=residuals, c='indigo', alpha=0.6)
plt.title('Residuals vs Predicted Values', fontsize=17)
plt.xlabel('Predicted Prices $\hat{y}_i$', fontsize=14)
plt.ylabel('Residuals', fontsize=14)
plt.show()

resid_mean = residuals.mean()
resid_skew = residuals.skew()

sns.displot(residuals, kde=True, color='indigo', aspect=2)
plt.title(f'Original Model Residuals: Skew ({resid_skew:.2f}) Mean ({resid_mean:.2f})')
plt.show()

In [ ]:
tgt_skew = data['PRICE'].skew()
sns.displot(data['PRICE'], kde=True, color='green', aspect=2)
plt.title(f'Normal Prices. Skew is {tgt_skew:.4f}')
plt.show()

y_log = np.log(data['PRICE'])
sns.displot(y_log, kde=True, aspect=2)
plt.title(f'Log Prices. Skew is {y_log.skew():.4f}')
plt.show()

plt.figure(dpi=150)
plt.scatter(data.PRICE, np.log(data.PRICE), color='seagreen', alpha=0.7)
plt.title('Mapping the Original Price to a Log Price')
plt.ylabel('Log Price')
plt.xlabel('Actual $ Price in $000s')
plt.show()

In [ ]:
new_target = np.log(data['PRICE'])
features = data.drop('PRICE', axis=1)

X_train_log, X_test_log, log_y_train, log_y_test = train_test_split(
    features, 
    new_target, 
    test_size=0.2, 
    random_state=10
)

log_regr = LinearRegression()
log_regr.fit(X_train_log, log_y_train)
log_rsquared = log_regr.score(X_train_log, log_y_train)

log_predictions = log_regr.predict(X_train_log)
log_residuals = (log_y_train - log_predictions)

print(f'Log Model Training Data R-squared: {log_rsquared:.4f}')
df_coef = pd.DataFrame(data=log_regr.coef_, index=X_train_log.columns, columns=['coef'])
display(df_coef)

In [ ]:
plt.figure(dpi=100)
plt.scatter(x=log_y_train, y=log_predictions, c='navy', alpha=0.6)
plt.plot(log_y_train, log_y_train, color='cyan')
plt.title(f'Actual vs Predicted Log Prices (R-Squared {log_rsquared:.2f})', fontsize=14)
plt.xlabel('Actual Log Prices $y_i$')
plt.ylabel('Predicted Log Prices $\hat{y}_i$')
plt.show()

plt.figure(dpi=100)
plt.scatter(x=log_predictions, y=log_residuals, c='navy', alpha=0.6)
plt.title('Residuals vs Fitted Values for Log Prices', fontsize=14)
plt.xlabel('Predicted Log Prices $\hat{y}_i$')
plt.ylabel('Residuals')
plt.show()

log_resid_mean = log_residuals.mean()
log_resid_skew = log_residuals.skew()

sns.displot(log_residuals, kde=True, color='navy', aspect=2)
plt.title(f'Log price model: Residuals Skew ({log_resid_skew:.2f}) Mean ({log_resid_mean:.2f})')
plt.show()

print(f'Original Model Test Data R-squared: {regr.score(X_test, y_test):.4f}')
print(f'Log Model Test Data R-squared: {log_regr.score(X_test_log, log_y_test):.4f}')

In [ ]:
features_base = data.drop(['PRICE'], axis=1)
average_vals = features_base.mean().values
property_stats = pd.DataFrame(data=average_vals.reshape(1, len(features_base.columns)), columns=features_base.columns)

next_to_river = True
nr_rooms = 8
students_per_classroom = 20 
distance_to_town = 5
pollution = data.NOX.quantile(q=0.75)       
amount_of_poverty = data.LSTAT.quantile(q=0.25) 

property_stats['RM'] = nr_rooms
property_stats['PTRATIO'] = students_per_classroom
property_stats['DIS'] = distance_to_town
property_stats['CHAS'] = 1 if next_to_river else 0
property_stats['NOX'] = pollution
property_stats['LSTAT'] = amount_of_poverty

log_estimate = log_regr.predict(property_stats)[0]
dollar_est = np.exp(log_estimate) * 1000

print(f'The log price estimate is: {log_estimate:.4f}')
print(f'The target property estimated market value is: ${dollar_est:,.2f}')